# Qwen Model Inference Notebook

This notebook loads and runs the Qwen 3.5-4B model from local files.

## 1. Import Required Libraries

Import necessary libraries including transformers, torch, os, and pathlib for model loading and file path handling.

In [8]:
import os
import sys
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path
import sentencepiece

## 2. Set Model File Path

Define the model file path using absolute or relative paths. Use os.path or pathlib to construct the correct path to the Qwen model directory.

In [11]:
# Set the base directory
BASE_DIR = "/home/yesoytur/APilus"

# Add backend to Python path
sys.path.append(os.path.join(BASE_DIR, 'backend'))

# Model path - pointing to the actual snapshot directory with model files
MODEL_LOCAL_PATH = os.path.join(BASE_DIR, "models/qwen/models--Qwen--Qwen3.5-4B/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a")

# Check if path exists
if os.path.exists(MODEL_LOCAL_PATH):
    print(f"Model path found: {MODEL_LOCAL_PATH}")
    # List files to verify
    print(f"Model files: {os.listdir(MODEL_LOCAL_PATH)}")
else:
    print(f"Model path not found: {MODEL_LOCAL_PATH}")
    print("Falling back to downloading from Hugging Face")
    MODEL_LOCAL_PATH = "Qwen/Qwen3.5-4B"

Model path found: /home/yesoytur/APilus/models/qwen/models--Qwen--Qwen3.5-4B/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a
Model files: ['chat_template.jinja', 'config.json', 'tokenizer_config.json', 'model.safetensors.index.json', 'model.safetensors-00001-of-00002.safetensors', 'model.safetensors-00002-of-00002.safetensors', 'tokenizer.json']


## 3. Load the Qwen Model

Use the transformers library to load the Qwen model from the specified file path using AutoModelForCausalLM and AutoTokenizer.

In [12]:
print(f"Loading Qwen Model from: {MODEL_LOCAL_PATH}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_LOCAL_PATH, 
    trust_remote_code=True
)

# Set pad token if not exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_LOCAL_PATH,
    torch_dtype=torch.float16, 
    device_map="auto",
    trust_remote_code=True
)

# Configure model
model.config.pad_token_id = tokenizer.pad_token_id

print("Model successfully loaded!")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading Qwen Model from: /home/yesoytur/APilus/models/qwen/models--Qwen--Qwen3.5-4B/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Model successfully loaded!


## 4. Run Inference

Execute inference on sample text inputs using the loaded model and tokenizer to generate predictions.

In [19]:
import torch

def generate_answer(user_prompt):
    # 1. Update the system prompt to explicitly prevent tool hallucination
    messages = [
        {"role": "system", "content": "You are a helpful conversational assistant. Respond directly to the user without using any tools."},
        {"role": "user", "content": user_prompt}
    ]
    
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # 2. Dynamically gather all possible stopping tokens for modern models
    terminators = [tokenizer.eos_token_id]
    for stop_token in ["<|eot_id|>", "<|im_end|>", "<|end_of_text|>"]:
        if stop_token in tokenizer.get_vocab():
            terminators.append(tokenizer.convert_tokens_to_ids(stop_token))

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.0, # Lowered from 1.1 to prevent format breaking
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=terminators # Pass the list of terminators instead of just one
        )
    
    # Get only the newly generated part
    input_length = model_inputs.input_ids.shape[1]
    generated_ids = generated_ids[:, input_length:]

    response = tokenizer.decode(
        generated_ids[0], 
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )
    
    # 3. Add the missing tool response tokens to the cleanup list
    artifacts = [
        # Code/tool tokens
        "<|fim_prefix|>", "<|file_sep|>", "<|fim_middle|>", "<|fim_suffix|>",
        "<|repo_name|>", "<tool_call>", "</tool_call>", "<tool_response>", "</tool_response>",
        "<think>", "</think>",
        # Chat tokens
        "<|im_start|>", "<|im_end|>", "<|system|>", "<|user|>", "<|assistant|>",
        # Other special tokens
        "<|endoftext|>", "[INST]", "[/INST]", "<s>", "</s>", "<pad>", "<unk>"
    ]
    
    for artifact in artifacts:
        response = response.replace(artifact, " ")
    
    # Clean up multiple spaces and whitespace
    response = " ".join(response.split())
    response = response.strip()
    
    return response if response else "I'm having trouble generating a response. Please try again."

# Test the model
user_prompt = "Hello, can you tell me about yourself?"
response = generate_answer(user_prompt)
print(f"User: {user_prompt}")
print(f"\nAssistant: {response}")

User: Hello, can you tell me about yourself?

Assistant: I'm having trouble generating a response. Please try again.


## 5. Additional Examples

Try different prompts to test the model.

In [14]:
# Example 1: Simple question
prompt1 = "What is the capital of France?"
response1 = generate_answer(prompt1)
print(f"Q: {prompt1}")
print(f"A: {response1}\n")

# Example 2: More complex question
prompt2 = "Explain quantum computing in simple terms."
response2 = generate_answer(prompt2)
print(f"Q: {prompt2}")
print(f"A: {response2}\n")

# You can add your own prompts here
# prompt3 = "Your custom prompt here"
# response3 = generate_answer(prompt3)
# print(f"Q: {prompt3}")
# print(f"A: {response3}")

Q: What is the capital of France?
A: </tool_call><|fim_prefix|><tool_response><|fim_middle|></tool_response></tool_response><|fim_pad|><|fim_suffix|><|repo_name|><|fim_pad|><|file_sep|><think><|fim_suffix|><|fim_suffix|><|fim_suffix|><|fim_suffix|><|repo_name|><think></tool_call><|fim_prefix|><tool_response><|fim_middle|>

Q: Explain quantum computing in simple terms.
A: <|fim_pad|><|repo_name|><|repo_name|><|fim_prefix|><|fim_middle|><tool_call><|fim_suffix|><|fim_prefix|></tool_call><|fim_middle|><|fim_suffix|><|file_sep|></tool_response><tool_response><|fim_suffix|></tool_response>

